In [1]:
!pip install -q ultralytics scikit-learn pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 26.2 MB/s eta 0:00:00


In [2]:
import os
import cv2
import json
import yaml
import math
import random
import zipfile
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from ultralytics import YOLO

from google.colab import drive

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


## 1. 원본 데이터 압축 해제

In [4]:
dataset_path = '/content/drive/MyDrive/ai09-level1-project.zip'
output_dir = '/content/ai09-project01/'
os.makedirs(output_dir, exist_ok=True)

with zipfile.ZipFile(dataset_path) as zip_file:
    zip_file.extractall(path=output_dir)

data_root = "/content/ai09-project01/sprint_ai_project1_data"
train_image_dir = os.path.join(data_root, "train_images")
train_annotation_dir = os.path.join(data_root, "train_annotations")
test_image_dir = os.path.join(data_root, "test_images")

print("Train image dir:", train_image_dir)
print("Train annotation dir:", train_annotation_dir)
print("Test image dir:", test_image_dir)

Train image dir: /content/ai09-project01/sprint_ai_project1_data/train_images
Train annotation dir: /content/ai09-project01/sprint_ai_project1_data/train_annotations
Test image dir: /content/ai09-project01/sprint_ai_project1_data/test_images


## 2. annotation 파싱 / YOLO 변환

In [5]:
def safe_get(d, keys, default=None):
    for k in keys:
        if k in d and d[k] not in [None, ""]:
            return d[k]
    return default

def get_image_files(image_dir, exts=(".png", ".jpg", ".jpeg")):
    image_files = sorted([
        f for f in os.listdir(image_dir)
        if f.lower().endswith(exts)
    ])
    print("Number of image files:", len(image_files))
    print("Sample image files:", image_files[:5])
    return image_files

def build_images_df(image_dir, image_files):
    rows = []

    for file_name in image_files:
        img_path = os.path.join(image_dir, file_name)
        img = cv2.imread(img_path)

        if img is None:
            rows.append({
                "file_name": file_name,
                "width": None,
                "height": None,
                "channel": None,
                "is_broken": True
            })
            continue

        h, w, c = img.shape
        rows.append({
            "file_name": file_name,
            "width": w,
            "height": h,
            "channel": c,
            "is_broken": False
        })

    images_df = pd.DataFrame(rows)
    images_df["image_key"] = images_df["file_name"].apply(lambda x: os.path.splitext(x)[0])
    return images_df

def parse_annotation_json(json_path, image_folder_name=None):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    rows = []

    if isinstance(data, dict) and "annotations" in data:
        annotations = data.get("annotations", [])
        images = data.get("images", [])
        categories = data.get("categories", [])

        image_info = images[0] if len(images) > 0 else {}
        image_id = safe_get(image_info, ["id"], default=None)
        file_name = safe_get(image_info, ["file_name"], default=None)

        category_map = {}
        for cat in categories:
            cat_id = safe_get(cat, ["id"], default=None)
            cat_name = safe_get(cat, ["name"], default=None)
            category_map[cat_id] = cat_name

        for ann in annotations:
            bbox = safe_get(ann, ["bbox"], default=None)
            category_id = safe_get(ann, ["category_id"], default=None)

            rows.append({
                "image_key": image_folder_name,
                "json_file": os.path.basename(json_path),
                "file_name_from_json": file_name,
                "image_id": image_id,
                "bbox": bbox,
                "category_id": category_id,
                "category_name": category_map.get(category_id, None),
                "dl_idx": safe_get(image_info, ["dl_idx"]),
                "dl_name": safe_get(image_info, ["dl_name"]),
                "drug_shape": safe_get(image_info, ["drug_shape"]),
                "color_class1": safe_get(image_info, ["color_class1"]),
                "color_class2": safe_get(image_info, ["color_class2"]),
                "line_front": safe_get(image_info, ["line_front"]),
                "line_back": safe_get(image_info, ["line_back"]),
                "print_front": safe_get(image_info, ["print_front"]),
                "print_back": safe_get(image_info, ["print_back"]),
                "json_path": json_path
            })
        return rows

    raise ValueError(f"Unsupported annotation format: {json_path}")

def build_annotations_df(annotation_root):
    rows = []

    for root, dirs, files in os.walk(annotation_root):
        json_files = [f for f in files if f.lower().endswith(".json")]
        for jf in json_files:
            json_path = os.path.join(root, jf)
            rel_dir = os.path.relpath(root, annotation_root)
            dir_parts = rel_dir.split(os.sep)

            json_stem = os.path.splitext(jf)[0]
            image_key = json_stem

            try:
                parsed_rows = parse_annotation_json(json_path, image_folder_name=image_key)
                for row in parsed_rows:
                    row["relative_dir"] = rel_dir
                    row["dir_depth"] = len(dir_parts)
                    rows.append(row)
            except Exception as e:
                rows.append({
                    "image_key": image_key,
                    "json_file": jf,
                    "file_name_from_json": None,
                    "image_id": None,
                    "bbox": None,
                    "category_id": None,
                    "category_name": None,
                    "dl_idx": None,
                    "dl_name": None,
                    "json_path": json_path,
                    "relative_dir": rel_dir,
                    "dir_depth": len(dir_parts),
                    "parse_error": str(e)
                })

    annotations_df = pd.DataFrame(rows)
    print("annotations_df shape:", annotations_df.shape)
    return annotations_df

In [6]:
def convert_loaded_annotations_to_yolo(
    images_df,
    annotations_df,
    train_image_dir,
    output_root,
    class_source="auto",
    copy_images=True,
    split_name="train"
):
    def is_valid_bbox(bbox):
        return isinstance(bbox, list) and len(bbox) == 4 and bbox[2] > 0 and bbox[3] > 0

    def xywh_to_yolo(img_w, img_h, bbox):
        x, y, w, h = bbox
        x_center = (x + w / 2.0) / img_w
        y_center = (y + h / 2.0) / img_h
        w = w / img_w
        h = h / img_h
        return x_center, y_center, w, h

    def choose_class_series(df, class_source):
        if class_source == "category_name" and "category_name" in df.columns:
            return df["category_name"]
        elif class_source == "dl_name" and "dl_name" in df.columns:
            return df["dl_name"]
        elif class_source == "category_id" and "category_id" in df.columns:
            return df["category_id"].astype(str)
        elif class_source == "dl_idx" and "dl_idx" in df.columns:
            return df["dl_idx"].astype(str)
        elif class_source == "auto":
            if "category_name" in df.columns and df["category_name"].notna().sum() > 0:
                return df["category_name"]
            elif "dl_name" in df.columns and df["dl_name"].notna().sum() > 0:
                return df["dl_name"]
            elif "category_id" in df.columns and df["category_id"].notna().sum() > 0:
                return df["category_id"].astype(str)
            elif "dl_idx" in df.columns and df["dl_idx"].notna().sum() > 0:
                return df["dl_idx"].astype(str)
            else:
                raise ValueError("No valid class column found in annotations_df.")
        else:
            raise ValueError(f"Invalid class_source: {class_source}")

    image_out_dir = os.path.join(output_root, "images", split_name)
    label_out_dir = os.path.join(output_root, "labels", split_name)
    os.makedirs(image_out_dir, exist_ok=True)
    os.makedirs(label_out_dir, exist_ok=True)

    merged_df = annotations_df.copy()
    class_series = choose_class_series(merged_df, class_source).fillna("UNKNOWN").astype(str)
    unique_classes = sorted(class_series.unique().tolist())
    class_to_id = {cls_name: idx for idx, cls_name in enumerate(unique_classes)}

    image_meta = images_df.set_index("image_key")[["file_name", "width", "height"]].to_dict("index")
    grouped = merged_df.groupby("image_key")

    valid_image_count = 0
    valid_box_count = 0

    for image_key, group in grouped:
        if image_key not in image_meta:
            continue

        file_name = image_meta[image_key]["file_name"]
        img_w = image_meta[image_key]["width"]
        img_h = image_meta[image_key]["height"]

        if pd.isna(img_w) or pd.isna(img_h):
            continue

        label_lines = []
        for _, row in group.iterrows():
            bbox = row.get("bbox", None)
            if not is_valid_bbox(bbox):
                continue

            cls_name = str(choose_class_series(group, class_source).loc[row.name]) if row.name in group.index else None
            if cls_name is None:
                continue

            cls_id = class_to_id[cls_name]
            x_c, y_c, w, h = xywh_to_yolo(img_w, img_h, bbox)
            label_lines.append(f"{cls_id} {x_c:.6f} {y_c:.6f} {w:.6f} {h:.6f}")
            valid_box_count += 1

        if len(label_lines) == 0:
            continue

        src_img_path = os.path.join(train_image_dir, file_name)
        dst_img_path = os.path.join(image_out_dir, file_name)
        dst_lbl_path = os.path.join(label_out_dir, f"{image_key}.txt")

        if copy_images and os.path.exists(src_img_path):
            shutil.copy2(src_img_path, dst_img_path)

        with open(dst_lbl_path, "w", encoding="utf-8") as f:
            f.write("\n".join(label_lines))

        valid_image_count += 1

    names_dict = {idx: name for name, idx in class_to_id.items()}
    yaml_path = os.path.join(output_root, "dataset.yaml")
    yaml_data = {
        "path": output_root,
        "train": f"images/{split_name}",
        "val": f"images/{split_name}",
        "names": names_dict
    }

    with open(yaml_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(yaml_data, f, sort_keys=False, allow_unicode=True)

    print(f"YOLO dataset created at: {output_root}")
    print("Valid images:", valid_image_count)
    print("Valid boxes:", valid_box_count)
    print("Number of classes:", len(class_to_id))

    return {
        "image_dir": image_out_dir,
        "label_dir": label_out_dir,
        "yaml_path": yaml_path,
        "class_to_id": class_to_id
    }


def make_yolo_kfold_dataset(
    pairs,
    image_dir,
    label_dir,
    output_root,
    class_to_id,
    n_splits=5,
    random_state=42,
    shuffle=True
):
    os.makedirs(output_root, exist_ok=True)

    names_dict = {idx: name for name, idx in class_to_id.items()}
    indices = list(range(len(pairs)))
    kf = KFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)

    fold_yaml_paths = []

    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(indices), start=1):
        fold_root = os.path.join(output_root, f"fold_{fold_idx}")
        fold_train_img_dir = os.path.join(fold_root, "images", "train")
        fold_val_img_dir = os.path.join(fold_root, "images", "val")
        fold_train_lbl_dir = os.path.join(fold_root, "labels", "train")
        fold_val_lbl_dir = os.path.join(fold_root, "labels", "val")

        os.makedirs(fold_train_img_dir, exist_ok=True)
        os.makedirs(fold_val_img_dir, exist_ok=True)
        os.makedirs(fold_train_lbl_dir, exist_ok=True)
        os.makedirs(fold_val_lbl_dir, exist_ok=True)

        for idx in train_idx:
            img_file, lbl_file = pairs[idx]
            shutil.copy2(os.path.join(image_dir, img_file), os.path.join(fold_train_img_dir, img_file))
            shutil.copy2(os.path.join(label_dir, lbl_file), os.path.join(fold_train_lbl_dir, lbl_file))

        for idx in val_idx:
            img_file, lbl_file = pairs[idx]
            shutil.copy2(os.path.join(image_dir, img_file), os.path.join(fold_val_img_dir, img_file))
            shutil.copy2(os.path.join(label_dir, lbl_file), os.path.join(fold_val_lbl_dir, lbl_file))

        yaml_path = os.path.join(fold_root, "dataset.yaml")
        yaml_data = {
            "path": fold_root,
            "train": "images/train",
            "val": "images/val",
            "names": names_dict
        }
        with open(yaml_path, "w", encoding="utf-8") as f:
            yaml.safe_dump(yaml_data, f, sort_keys=False, allow_unicode=True)

        fold_yaml_paths.append(yaml_path)

    print("Created fold yaml files:")
    for p in fold_yaml_paths:
        print(" -", p)

    return fold_yaml_paths

## 3. YOLO 형식 데이터셋 생성 + K-Fold 분할

In [7]:
from sklearn.model_selection import KFold

def convert_loaded_annotations_to_yolo(
    images_df,
    annotations_df,
    train_image_dir,
    output_root,
    class_source="auto",
    copy_images=True,
    split_name="train"
):
    def is_valid_bbox(bbox):
        return isinstance(bbox, list) and len(bbox) == 4 and bbox[2] > 0 and bbox[3] > 0

    def xywh_to_yolo(img_w, img_h, bbox):
        x, y, w, h = bbox
        x_center = (x + w / 2.0) / img_w
        y_center = (y + h / 2.0) / img_h
        w = w / img_w
        h = h / img_h
        return x_center, y_center, w, h

    def choose_class_series(df, class_source):
        if class_source == "category_name" and "category_name" in df.columns:
            return df["category_name"]
        elif class_source == "dl_name" and "dl_name" in df.columns:
            return df["dl_name"]
        elif class_source == "category_id" and "category_id" in df.columns:
            return df["category_id"].astype(str)
        elif class_source == "dl_idx" and "dl_idx" in df.columns:
            return df["dl_idx"].astype(str)
        elif class_source == "auto":
            if "category_name" in df.columns and df["category_name"].notna().sum() > 0:
                return df["category_name"]
            elif "dl_name" in df.columns and df["dl_name"].notna().sum() > 0:
                return df["dl_name"]
            elif "category_id" in df.columns and df["category_id"].notna().sum() > 0:
                return df["category_id"].astype(str)
            elif "dl_idx" in df.columns and df["dl_idx"].notna().sum() > 0:
                return df["dl_idx"].astype(str)
            else:
                raise ValueError("No valid class column found in annotations_df.")
        else:
            raise ValueError(f"Invalid class_source: {class_source}")

    image_out_dir = os.path.join(output_root, "images", split_name)
    label_out_dir = os.path.join(output_root, "labels", split_name)
    os.makedirs(image_out_dir, exist_ok=True)
    os.makedirs(label_out_dir, exist_ok=True)

    merged_df = annotations_df.copy()
    class_series = choose_class_series(merged_df, class_source).fillna("UNKNOWN").astype(str)
    unique_classes = sorted(class_series.unique().tolist())
    class_to_id = {cls_name: idx for idx, cls_name in enumerate(unique_classes)}

    image_meta = images_df.set_index("image_key")[["file_name", "width", "height"]].to_dict("index")
    grouped = merged_df.groupby("image_key")

    valid_image_count = 0
    valid_box_count = 0

    for image_key, group in grouped:
        if image_key not in image_meta:
            continue

        file_name = image_meta[image_key]["file_name"]
        img_w = image_meta[image_key]["width"]
        img_h = image_meta[image_key]["height"]

        if pd.isna(img_w) or pd.isna(img_h):
            continue

        label_lines = []
        for _, row in group.iterrows():
            bbox = row.get("bbox", None)
            if not is_valid_bbox(bbox):
                continue

            cls_name = str(choose_class_series(group, class_source).loc[row.name]) if row.name in group.index else None
            if cls_name is None:
                continue

            cls_id = class_to_id[cls_name]
            x_c, y_c, w, h = xywh_to_yolo(img_w, img_h, bbox)
            label_lines.append(f"{cls_id} {x_c:.6f} {y_c:.6f} {w:.6f} {h:.6f}")
            valid_box_count += 1

        if len(label_lines) == 0:
            continue

        src_img_path = os.path.join(train_image_dir, file_name)
        dst_img_path = os.path.join(image_out_dir, file_name)
        dst_lbl_path = os.path.join(label_out_dir, f"{image_key}.txt")

        if copy_images and os.path.exists(src_img_path):
            shutil.copy2(src_img_path, dst_img_path)

        with open(dst_lbl_path, "w", encoding="utf-8") as f:
            f.write("\n".join(label_lines))

        valid_image_count += 1

    names_dict = {idx: name for name, idx in class_to_id.items()}
    yaml_path = os.path.join(output_root, "dataset.yaml")
    yaml_data = {
        "path": output_root,
        "train": f"images/{split_name}",
        "val": f"images/{split_name}",
        "names": names_dict
    }

    with open(yaml_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(yaml_data, f, sort_keys=False, allow_unicode=True)

    print(f"YOLO dataset created at: {output_root}")
    print("Valid images:", valid_image_count)
    print("Valid boxes:", valid_box_count)
    print("Number of classes:", len(class_to_id))

    return {
        "image_dir": image_out_dir,
        "label_dir": label_out_dir,
        "yaml_path": yaml_path,
        "class_to_id": class_to_id
    }


def make_yolo_kfold_dataset(
    pairs,
    image_dir,
    label_dir,
    output_root,
    class_to_id,
    n_splits=5,
    random_state=42,
    shuffle=True
):
    os.makedirs(output_root, exist_ok=True)

    names_dict = {idx: name for name, idx in class_to_id.items()}
    indices = list(range(len(pairs)))
    kf = KFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)

    fold_yaml_paths = []

    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(indices), start=1):
        fold_root = os.path.join(output_root, f"fold_{fold_idx}")
        fold_train_img_dir = os.path.join(fold_root, "images", "train")
        fold_val_img_dir = os.path.join(fold_root, "images", "val")
        fold_train_lbl_dir = os.path.join(fold_root, "labels", "train")
        fold_val_lbl_dir = os.path.join(fold_root, "labels", "val")

        os.makedirs(fold_train_img_dir, exist_ok=True)
        os.makedirs(fold_val_img_dir, exist_ok=True)
        os.makedirs(fold_train_lbl_dir, exist_ok=True)
        os.makedirs(fold_val_lbl_dir, exist_ok=True)

        for idx in train_idx:
            img_file, lbl_file = pairs[idx]
            shutil.copy2(os.path.join(image_dir, img_file), os.path.join(fold_train_img_dir, img_file))
            shutil.copy2(os.path.join(label_dir, lbl_file), os.path.join(fold_train_lbl_dir, lbl_file))

        for idx in val_idx:
            img_file, lbl_file = pairs[idx]
            shutil.copy2(os.path.join(image_dir, img_file), os.path.join(fold_val_img_dir, img_file))
            shutil.copy2(os.path.join(label_dir, lbl_file), os.path.join(fold_val_lbl_dir, lbl_file))

        yaml_path = os.path.join(fold_root, "dataset.yaml")
        yaml_data = {
            "path": fold_root,
            "train": "images/train",
            "val": "images/val",
            "names": names_dict
        }
        with open(yaml_path, "w", encoding="utf-8") as f:
            yaml.safe_dump(yaml_data, f, sort_keys=False, allow_unicode=True)

        fold_yaml_paths.append(yaml_path)

    print("Created fold yaml files:")
    for p in fold_yaml_paths:
        print(" -", p)

    return fold_yaml_paths

In [8]:
image_files = get_image_files(train_image_dir)
images_df = build_images_df(train_image_dir, image_files)
annotations_df = build_annotations_df(train_annotation_dir)

annotations_df = annotations_df.copy()
if "file_name_from_json" in annotations_df.columns:
    mask = annotations_df["file_name_from_json"].notna()
    annotations_df.loc[mask, "image_key"] = annotations_df.loc[mask, "file_name_from_json"].apply(
        lambda x: os.path.splitext(os.path.basename(x))[0]
    )

yolo_info = convert_loaded_annotations_to_yolo(
    images_df=images_df,
    annotations_df=annotations_df,
    train_image_dir=train_image_dir,
    output_root="/content/yolo_pill_dataset",
    class_source="auto",
    copy_images=True,
    split_name="train"
)

image_dir = yolo_info["image_dir"]
label_dir = yolo_info["label_dir"]
class_to_id = yolo_info["class_to_id"]

image_files = sorted([
    f for f in os.listdir(image_dir)
    if f.lower().endswith((".png", ".jpg", ".jpeg"))
])

pairs = []
for img_file in image_files:
    stem = os.path.splitext(img_file)[0]
    label_file = f"{stem}.txt"
    if os.path.exists(os.path.join(label_dir, label_file)):
        pairs.append((img_file, label_file))

print("Number of image-label pairs:", len(pairs))

fold_yaml_paths = make_yolo_kfold_dataset(
    pairs=pairs,
    image_dir=image_dir,
    label_dir=label_dir,
    output_root="/content/yolo_pill_dataset_kfold",
    class_to_id=class_to_id,
    n_splits=5,
    random_state=42,
    shuffle=True
)

Number of image files: 232
Sample image files: ['K-001900-016548-019607-029451_0_2_0_2_70_000_200.png', 'K-001900-016548-019607-029451_0_2_0_2_75_000_200.png', 'K-001900-016548-019607-029451_0_2_0_2_90_000_200.png', 'K-001900-016548-019607-033009_0_2_0_2_70_000_200.png', 'K-001900-016548-019607-033009_0_2_0_2_75_000_200.png']
annotations_df shape: (763, 19)
YOLO dataset created at: /content/yolo_pill_dataset
Valid images: 232
Valid boxes: 763
Number of classes: 56
Number of image-label pairs: 232
Created fold yaml files:
 - /content/yolo_pill_dataset_kfold/fold_1/dataset.yaml
 - /content/yolo_pill_dataset_kfold/fold_2/dataset.yaml
 - /content/yolo_pill_dataset_kfold/fold_3/dataset.yaml
 - /content/yolo_pill_dataset_kfold/fold_4/dataset.yaml
 - /content/yolo_pill_dataset_kfold/fold_5/dataset.yaml


## 4. Ultralytics baseline 설정

아래 셀은
1. 전체 fold를 순회하며 학습/검증하고
2. fold별 성능을 표로 정리한 뒤
3. 마지막에 전체 train images로 최종 학습

을 수행하도록 구성했다.


In [9]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

MODEL_NAME = "yolo26m.pt"   # "yolo11n.pt" / "yolo26n.pt"
IMGSZ = 960
EPOCHS = 120
BATCH = 8
WORKERS = 2

KFOLD_PROJECT_NAME = f"ultralytics_{MODEL_NAME}_kfold_runs"
FINAL_PROJECT_NAME = f"ultralytics_{MODEL_NAME}_fulltrain_runs"

FINAL_RUN_NAME = f"fulltrain_{Path(MODEL_NAME).stem}_img{IMGSZ}_bs{BATCH}"

print("Model:", MODEL_NAME)
print("IMGSZ:", IMGSZ)
print("EPOCHS:", EPOCHS)
print("BATCH:", BATCH)
print("Number of folds:", len(fold_yaml_paths))
print("Full-train yaml:", yolo_info["yaml_path"])

Model: yolo26m.pt
IMGSZ: 960
EPOCHS: 120
BATCH: 8
Number of folds: 5
Full-train yaml: /content/yolo_pill_dataset/dataset.yaml


## 5. 전체 fold 학습 + 검증

각 fold마다 `best.pt`와 `last.pt`를 저장하고,  
학습 후 `best.pt` 기준으로 다시 `model.val()`을 수행해 fold별 mAP를 정리한다.


In [16]:
from ultralytics import YOLO
import pandas as pd

fold_run_records = []

for fold_idx, fold_yaml_path in enumerate(fold_yaml_paths, start=1):
    fold_root = os.path.dirname(fold_yaml_path)
    run_name = f"fold{fold_idx}_{Path(MODEL_NAME).stem}_img{IMGSZ}_bs{BATCH}"

    print("=" * 80)
    print(f"[Fold {fold_idx}/{len(fold_yaml_paths)}]")
    print("YAML:", fold_yaml_path)
    print("Run name:", run_name)

    model = YOLO(MODEL_NAME)

    train_results = model.train(
    data=fold_yaml_path,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    workers=WORKERS,
    seed=SEED,
    deterministic=True,
    pretrained=True,
    project=KFOLD_PROJECT_NAME,
    name=run_name,

    # optimizer / lr
    optimizer="auto",
    lr0=0.005,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3.0,

    # augmentation
    fliplr=0.5,
    flipud=0.15,
    degrees=20,
    translate=0.20,
    scale=0.50,
    shear=3.0,
    perspective=0.0005,

    hsv_h=0.01,
    hsv_s=0.50,
    hsv_v=0.50,

    mosaic=0.30,
    mixup=0.05,
    close_mosaic=10,

    # save / val
    plots=True,
    val=True,
    save=True,
    save_period=-1,
    verbose=True,
)

    best_ckpt = model.trainer.best
    last_ckpt = model.trainer.last

    print("Best checkpoint:", best_ckpt)
    print("Last checkpoint:", last_ckpt)

    best_model = YOLO(best_ckpt)
    metrics = best_model.val(
        data=fold_yaml_path,
        imgsz=IMGSZ,
        batch=BATCH,
        workers=WORKERS,
        plots=False,
        verbose=False
    )

    fold_run_records.append({
        "fold": fold_idx,
        "yaml_path": fold_yaml_path,
        "best_ckpt": str(best_ckpt),
        "last_ckpt": str(last_ckpt),
        "map50": float(metrics.box.map50),
        "map75": float(metrics.box.map75),
        "map50_95": float(metrics.box.map),
    })

fold_results_df = pd.DataFrame(fold_run_records)
fold_results_df

[Fold 1/5]
YAML: /content/yolo_pill_dataset_kfold/fold_1/dataset.yaml
Run name: fold1_yolo26m_img960_bs16
Ultralytics 8.4.26 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_pill_dataset_kfold/fold_1/dataset.yaml, degrees=20, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=120, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.15, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.01, hsv_s=0.5, hsv_v=0.5, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.05, mode=train, model=yolo26m.pt, momentum=0.937, mosaic=0.3, multi

KeyboardInterrupt: 

## 6. Fold 결과 요약

In [ ]:
display(fold_results_df)

summary_df = pd.DataFrame([{
    "map50_mean": fold_results_df["map50"].mean(),
    "map50_std": fold_results_df["map50"].std(ddof=1),
    "map75_mean": fold_results_df["map75"].mean(),
    "map75_std": fold_results_df["map75"].std(ddof=1),
    "map50_95_mean": fold_results_df["map50_95"].mean(),
    "map50_95_std": fold_results_df["map50_95"].std(ddof=1),
}])

display(summary_df)

## 7. 대표 fold의 validation 이미지 예측 시각화

최고 `mAP50:95`를 낸 fold의 `best.pt`를 사용해 validation 이미지 예측 결과를 확인한다.


In [ ]:
best_fold_row = fold_results_df.sort_values("map50_95", ascending=False).iloc[0]
best_fold = int(best_fold_row["fold"])
best_fold_yaml_path = best_fold_row["yaml_path"]
best_fold_ckpt = best_fold_row["best_ckpt"]
best_fold_root = os.path.dirname(best_fold_yaml_path)

print("Best fold:", best_fold)
print("Best fold ckpt:", best_fold_ckpt)

best_fold_model = YOLO(best_fold_ckpt)

val_image_dir = os.path.join(best_fold_root, "images", "val")
val_images = sorted([
    os.path.join(val_image_dir, f)
    for f in os.listdir(val_image_dir)
    if f.lower().endswith((".png", ".jpg", ".jpeg"))
])

sample_images = val_images[:5]
print("Number of val images:", len(val_images))
print("Sample images:", sample_images[:3])

In [ ]:
pred_results = best_fold_model.predict(
    source=sample_images,
    imgsz=IMGSZ,
    conf=0.25,
    iou=0.5,
    save=False,
    show=False,
    verbose=False
)

plt.figure(figsize=(14, 14))
for i, r in enumerate(pred_results, start=1):
    im_bgr = r.plot()
    im_rgb = cv2.cvtColor(im_bgr, cv2.COLOR_BGR2RGB)
    plt.subplot(len(pred_results), 1, i)
    plt.imshow(im_rgb)
    plt.axis("off")
    plt.title(os.path.basename(r.path))
plt.tight_layout()
plt.show()

## 8. 전체 train images로 최종 학습

K-Fold 실험으로 설정을 확인한 뒤, 전체 train images를 사용해 최종 모델을 학습한다.

참고:
- `yolo_info["yaml_path"]`는 train과 val이 모두 같은 `images/train`을 가리킨다.
- 따라서 아래 최종 학습은 **검증 성능 비교용이 아니라 최종 weight 확보용**으로 사용한다.
- 최종 제출용 추론은 여기서 저장된 `best.pt` 또는 `last.pt`를 사용하면 된다.


In [12]:
final_model = YOLO(MODEL_NAME)

final_train_results = final_model.train(
    data=yolo_info["yaml_path"],
    epochs=70,
    imgsz=IMGSZ,
    batch=BATCH,
    workers=WORKERS,
    seed=SEED,
    deterministic=True,
    pretrained=True,
    project=f"{KFOLD_PROJECT_NAME}_full_train_final",
    name=f"YOLO26-midium_full_train_final",

    # optimizer / lr
    optimizer="auto",
    lr0=0.005,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3.0,

    # augmentation
    fliplr=0.5,
    flipud=0.15,
    degrees=20,
    translate=0.20,
    scale=0.50,
    shear=3.0,
    perspective=0.0005,

    hsv_h=0.01,
    hsv_s=0.50,
    hsv_v=0.50,

    mosaic=0.30,
    mixup=0.05,
    close_mosaic=10,

    # save / val
    plots=True,
    val=True,
    save=True,
    save_period=-1,
    verbose=True,
)

print("Final best checkpoint:", final_model.trainer.best)
print("Final last checkpoint:", final_model.trainer.last)

Ultralytics 8.4.26 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_pill_dataset/dataset.yaml, degrees=20, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=70, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.15, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.01, hsv_s=0.5, hsv_v=0.5, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.05, mode=train, model=yolo26m.pt, momentum=0.937, mosaic=0.3, multi_scale=0.0, name=YOLO26-midium_full_train_final, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_m

## 9. 실험 메모

- fold별 결과는 `fold_results_df`
- 최종 전체 학습 checkpoint는 `final_model.trainer.best`, `final_model.trainer.last`
- test inference / submission 생성 시에는 보통 최종 전체 학습의 `best.pt`를 사용
